# Block preconditioner vs SANEPIC's Jacobi preconditioner

pysanepic solves (PᵀN⁻¹P) m = PᵀN⁻¹d with preconditioned conjugate gradients
(PCG). Two preconditioners are available:

* `"jacobi"`: the original SANEPIC choice. The I part is the exact diagonal of
  PᵀN⁻¹P; Q and U use 2/diag_I, which assumes a uniform coverage of
  polarization angles in each pixel.
* `"block"` (default): the inverse of the exact 3×3 I/Q/U block of PᵀN⁻¹P in
  each pixel, built from the actual angles, polarization efficiency and 1/f
  noise correlation. For white noise it is exactly the inverse of the binned-map
  covariance.

Both converge to the same map; this notebook compares how fast, on one day of
LiteBIRD simulation with and without HWP.

Q and U are solved only in pixels where they can be separated: at least 4 hits
and a polarization-angle coverage whose reciprocal condition number is at least
`min_pol_rcond` (default 1e-2). Without HWP, one day of a single T/B detector
pair leaves many pixels with a poor angle coverage, where only I is solved.

Requirements: `pysanepic` and a `litebird_sim` version providing
`Simulation.make_sanepic_gls_map` (until [litebird_sim#568](https://github.com/litebird/litebird_sim/pull/568)
is merged: `pip install git+https://github.com/litebird/litebird_sim@sanepic_gls`).

In [ ]:
import time

import astropy.time
import healpy as hp
import litebird_sim as lbs
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
telescope, channel = "LFT", "L4-140"
detlist = ["000_001_017_QB_140_T", "000_001_017_QB_140_B"]
duration_days = 1.0
nside = 64
chunk_s = 3600.0
imo = lbs.Imo(flatfile_location=lbs.PTEP_IMO_LOCATION)


def simulate(with_hwp):
    sim = lbs.Simulation(
        base_path="preconditioner_output",
        imo=imo,
        start_time=astropy.time.Time("2030-01-01T00:00:00"),
        duration_s=duration_days * 86400,
        random_seed=12345,
    )
    sim.set_instrument(
        lbs.InstrumentInfo.from_imo(imo, f"/releases/vPTEP/satellite/{telescope}/instrument_info")
    )
    dets = [
        lbs.DetectorInfo.from_imo(
            url=f"/releases/vPTEP/satellite/{telescope}/{channel}/{d}/detector_info", imo=imo
        )
        for d in detlist
    ]
    sim.set_scanning_strategy(imo_url="/releases/vPTEP/satellite/scanning_parameters/")
    sim.create_observations(detectors=dets)
    if with_hwp:
        sim.set_hwp(lbs.IdealHWP(sim.instrument.hwp_rpm * 2 * np.pi / 60))
    sim.prepare_pointings()
    ch = lbs.FreqChannelInfo.from_imo(
        url=f"/releases/vPTEP/satellite/{telescope}/{channel}/channel_info", imo=imo
    )
    sim.get_sky(
        parameters=lbs.SkyGenerationParams(
            make_cmb=True, make_fg=False, seed_cmb=1, apply_beam=True,
            units="K_CMB", output_type="map", nside=nside,
        ),
        channels=[ch],
        store_in_observation=True,
    )
    sim.fill_tods()
    sim.add_noise(noise_type="one_over_f")
    return sim

## Run both preconditioners on both simulations

The same stopping threshold is used for all runs.

In [ ]:
results = {}
for with_hwp in [True, False]:
    sim = simulate(with_hwp)
    for kind in ["jacobi", "block"]:
        t0 = time.time()
        res = sim.make_sanepic_gls_map(
            nside=nside, chunk_s=chunk_s, preconditioner=kind, append_to_report=False
        )
        label = f"{'HWP' if with_hwp else 'no HWP'}, {kind}"
        results[label] = (res, time.time() - t0)
        n_qu = np.sum(res.maps[1] != hp.UNSEEN)
        n_i = np.sum(res.maps[0] != hp.UNSEEN)
        print(f"{label:16s} {res.iterations:4d} iterations  {time.time() - t0:6.1f} s  "
              f"converged: {res.converged}  Q/U solved in {n_qu}/{n_i} pixels")

In [ ]:
plt.figure(figsize=(8, 5))
for label, (res, _) in results.items():
    plt.semilogy(res.residuals, label=label, ls="-" if "block" in label else "--")
plt.xlabel("iteration")
plt.ylabel(r"$|r|^2 / |b|^2$")
plt.legend()
plt.title("PCG convergence");

## Same solution

The converged maps agree to a small fraction of the noise level; the I monopole
is removed because 1/f noise leaves it (almost) unconstrained.

In [ ]:
for setup in ["HWP", "no HWP"]:
    a = results[f"{setup}, block"][0].maps
    b = results[f"{setup}, jacobi"][0].maps
    seen = (a[1] != hp.UNSEEN) & (b[1] != hp.UNSEEN)
    d = (a - b)[:, seen]
    d[0] -= d[0].mean()
    print(f"{setup:7s} max |block - jacobi| I/Q/U [uK]: " + " / ".join(f"{1e6 * np.abs(x).max():.2e}" for x in d))

With the HWP, every pixel is observed at many polarization angles: SANEPIC's
assumption of uniform angle coverage holds and both preconditioners converge
quickly, the block one with slightly fewer iterations. Without HWP the angle
coverage is limited, the Jacobi Q/U approximation is poor and the block
preconditioner needs about half the iterations. In poorly scanned data (the
small test simulation of `tests/test_vs_sanepic.py`) the gain is larger, 23 vs
312 iterations.

Without the `min_pol_rcond` criterion (`min_pol_rcond=0`, as in SANEPIC), Q and
U would also be "solved" in the pixels where they cannot be separated: the
result there is dominated by noise and the Jacobi PCG stalls.